# 1. Khai báo thư viện phục vụ phân tích

In [1]:
from pathlib import Path
import pandas as pd
import json
print(Path)
print(type(Path))

p = Path("../data/raw/2019-Oct.csv")
print(p)

<class 'pathlib.Path'>
<class 'type'>
..\data\raw\2019-Oct.csv


# 2. Nạp tập dữ liệu mẫu 100,000 dòng để phân tích hiệu năng

In [2]:
DATA_PATH = Path("../data/raw/2019-Oct.csv")
df = pd.read_csv(DATA_PATH)
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [3]:
df.head(5)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


# 3.Kiểm tra ma trận kích thước (Shape Analysis)

In [4]:
print(f"Số lượng dòng (Rows): {df.shape[0]}")
print(f"Số lượng cột (Columns): {df.shape[1]}")

Số lượng dòng (Rows): 42448764
Số lượng cột (Columns): 9


# 4.Kiểm tra cấu trúc tổng quan và kiểu dữ liệu hệ thống

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42448764 entries, 0 to 42448763
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 2.8+ GB


# 5. Phân tích giá trị thiếu (Missing Values)

In [6]:
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percentage
})
missing_df

,Missing Count,Percentage (%)
event_time,0,0.00
event_type,0,0.00
product_id,0,0.00
category_id,0,0.00
category_code,13515609,31.84
brand,6117080,14.41
price,0,0.00
user_id,0,0.00
user_session,2,0.00


# 6. Xử lý dữ liệu trùng lặp (Duplicates)

In [7]:
duplicate_count = df.duplicated().sum()
print(f"Số lượng bản ghi trùng lặp hoàn toàn trong tập mẫu: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Đã loại bỏ các bản ghi trùng lặp.")

Số lượng bản ghi trùng lặp hoàn toàn trong tập mẫu: 30220
Đã loại bỏ các bản ghi trùng lặp.


# 7.Chuẩn hóa kiểu dữ liệu Chuỗi thời gian (Datetime Conversion)

In [8]:
print(f"Kiểu dữ liệu gốc của cột event_time: {df['event_time'].dtype}")

# Ép kiểu sang Datetime dạng chuẩn chuỗi thời gian để tối ưu hóa truy vấn
df["event_time"] = pd.to_datetime(df["event_time"])
print(f"Kiểu dữ liệu sau khi biến đổi: {df['event_time'].dtype}")

Kiểu dữ liệu gốc của cột event_time: object
Kiểu dữ liệu sau khi biến đổi: datetime64[ns, UTC]


# 8.Kỹ nghệ đặc trưng (Feature Engineering) cho Tầng AI Agent

In [9]:
df["hour"] = df["event_time"].dt.hour.astype("int8")
df["day"] = df["event_time"].dt.day.astype("int8")
df["weekday"] = df["event_time"].dt.day_name().astype("category")
df["month"] = df["event_time"].dt.month.astype("int8")

# Xem nhanh các đặc trưng mới tạo
df[["event_time", "hour", "day", "weekday", "month"]].head()

,event_time,hour,day,weekday,month
0,2019-10-01 00:00:00+00:00,0,1,Tuesday,10
1,2019-10-01 00:00:00+00:00,0,1,Tuesday,10
2,2019-10-01 00:00:01+00:00,0,1,Tuesday,10
3,2019-10-01 00:00:01+00:00,0,1,Tuesday,10
4,2019-10-01 00:00:04+00:00,0,1,Tuesday,10


# 9. Đóng gói và lưu trữ tối ưu dưới định dạng Apache Parquet

In [10]:
import pandas as pd
import pyarrow as pa
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

Python: 3.12.13 (main, Mar 20 2026, 00:36:00) [MSC v.1944 64 bit (AMD64)]
Pandas: 2.2.3
PyArrow: 18.1.0


In [11]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Ghi dữ liệu xuống định dạng Parquet nén cột cao cấp
df.to_parquet(
    PROCESSED_DIR / "events.parquet",
    index=False,
    compression="snappy"
)
print("Đã xuất lưu trữ thành công tệp tin nền tảng events.parquet!")

Đã xuất lưu trữ thành công tệp tin nền tảng events.parquet!


In [13]:
import duckdb

duckdb.sql("""
SELECT *
FROM 'D:\Project\AI_Data_Intelligence_Platform\data\processed\events.parquet'
LIMIT 10
""").show()

┌──────────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────────────────────┬──────────┬─────────┬───────────┬──────────────────────────────────────┬──────┬──────┬─────────┬───────┐
│        event_time        │ event_type │ product_id │     category_id     │            category_code            │  brand   │  price  │  user_id  │             user_session             │ hour │ day  │ weekday │ month │
│ timestamp with time zone │  varchar   │   int64    │        int64        │               varchar               │ varchar  │ double  │   int64   │               varchar                │ int8 │ int8 │ varchar │ int8  │
├──────────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────────────────────┼──────────┼─────────┼───────────┼──────────────────────────────────────┼──────┼──────┼─────────┼───────┤
│ 2019-10-01 07:00:00+07   │ view       │   44600062 │ 2103807459595387724 │ NULL                                │ shiseido 

<>:5: SyntaxWarning: invalid escape sequence '\P'
<>:5: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_32836\1772287819.py:5: SyntaxWarning: invalid escape sequence '\P'
  FROM 'D:\Project\AI_Data_Intelligence_Platform\data\processed\events.parquet'
